<a href="https://colab.research.google.com/github/elisacapilli/ai-notebooks/blob/upload-notebooks/1_stat_Gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install --force-reinstall transformers vllm

In [ ]:
!pip install --upgrade numpy torch transformers vllm --no-cache-dir

In [ ]:
import numpy
import transformers
import vllm

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import numpy as np
import pandas as pd
import torch
import random
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, AutoConfig, TrainingArguments, Trainer, AutoModelForCausalLM, pipeline
tqdm.pandas()
import os
os.environ["WANDB_MODE"] = "disabled"

from vllm import LLM, SamplingParams

# import outlines
from pydantic import BaseModel
from lmformatenforcer import RegexParser
from lmformatenforcer.integrations.transformers import build_transformers_prefix_allowed_tokens_fn
from lmformatenforcer.integrations.vllm import build_vllm_token_enforcer_tokenizer_data, build_vllm_logits_processor

CACHE = '/content/.cache/huggingface/'

from huggingface_hub import login
login(token="HF_TOKEN")

import warnings
warnings.filterwarnings('ignore')

In [ ]:
model_id = 'google/gemma-2-2b-it'

llm = LLM(
    model=model_id,
    tokenizer=model_id,
    tokenizer_mode="mistral" if "mistral" in model_id else "auto",
    trust_remote_code=True,
    dtype=torch.float16,
    max_model_len=2048*2,
    tensor_parallel_size=1,
    download_dir=CACHE,
)

tokenizer =  build_vllm_token_enforcer_tokenizer_data(llm)

In [ ]:
from IPython.display import display

df = pd.read_excel("df_facct.xlsx")

In [ ]:
value_counts = df['statement_text'].value_counts()

frequent_statements = value_counts[value_counts >= 5].index

df_filtered = df[df['statement_text'].isin(frequent_statements)]

In [ ]:
def remap_truthfulness_scale(df, col='truthfulness_value'):
    scale_map = {
        -2: 0,
        -1: 1,
        0: 2,
        1: 3,
        2: 5
    }

    df[col] = df[col].map(scale_map)
    return df

df_filtered = remap_truthfulness_scale(df_filtered)

In [ ]:
selected_statement = df_filtered['statement_text'].sample(1).iloc[0]
df_single_statement = df_filtered[df_filtered['statement_text'] == selected_statement]

print(f"Statement selezionato:\n{selected_statement}\n")
display(df_single_statement)

In [ ]:
df_filtered = df_single_statement.dropna(subset=['truthfulness_value'])

selected_rows = df_filtered.sample(n=5, random_state=42)  # Se desideri random puro, rimuovi random_state

display(selected_rows)

In [ ]:
truthfulness_values_list = selected_rows['truthfulness_value'].astype(int).tolist()

print(truthfulness_values_list)

In [ ]:
def generate_prompt_fstring(row):
    if pd.isna(row['truthfulness_value']) or pd.isna(row['statement_text']):
        print("Errore: La riga contiene valori NaN.")
        return None

    worker_truthfulness_values = row['truthfulness_value']

    if isinstance(truthfulness_values_list, list):
        if len(truthfulness_values_list) != 5:
            print("Errore: La lista di valutazioni non contiene esattamente 5 elementi.")
            return None
        worker_votes = [f"w{i+1} ha votato {value}" for i, value in enumerate(truthfulness_values_list)]
    else:
        worker_votes = [f"w1 ha votato {truthfulness_values_list}"]

    worker_votes_str = "\n".join(worker_votes)

    print(f"Lista delle valutazioni: {worker_votes}")
    print(f"Stringa delle valutazioni dei worker:\n{worker_votes_str}")

    if not row['statement_text']:
        print("Errore: La colonna 'statement_text' è vuota.")
        return None

    prompt = f"""
        You are a professional fact-checking AI. You will be presented with a statement along with five evaluations provided by anonymous human workers through crowdsourcing.
        These workers assess the factual accuracy of the statement using the PolitiFact scale.

        Each worker assigns an evaluation to the statement, and your task is to evaluate the truthfulness of the statement using real-world facts and verified evidence.
        While you should consider the workers' evaluations, keep in mind that they may be inaccurate, biased, or may not fully reflect the true factuality of the statement.

        PolitiFact Factuality Scale (integer values only):
        0 = Pants on Fire — statement completely false and absurd
        1 = False — statement entirely incorrect
        2 = Mostly False — statement contains significant falsehoods or misleading elements
        3 = Half True — statement mixes truth and falsehood
        4 = Mostly True — statement mostly accurate with minor errors
        5 = True — statementfully accurate and supported by verifiable facts

        Your task:
        - Evaluate the truthfulness of the statement based on factual, verified information.
        - Consider the evaluations provided by the workers: {worker_votes_str}.
        - Apply critical judgment, considering that the evaluations are provided in sequence.
        - Based on the evaluations, provide a single integer answer from 0 to 5, representing your factuality rating for the statement.
        - Your answer should consider all the five evaluations.
        - Provide no explanations or additional text.

        Statement: "{row['statement_text']}"
        Worker Evaluations:
        {worker_votes_str}
        Answer:
        """

    return prompt
    print(f"Prompt generato:\n{prompt}")
selected_row = df_single_statement.iloc[0]
prompt = generate_prompt_fstring(selected_row)

if prompt:
    print("\nPrompt generato con successo!")
else:
    print("\nErrore: il prompt non è stato generato correttamente.")



In [ ]:
def get_model_response(prompt):
    logits_processor = build_vllm_logits_processor(tokenizer, RegexParser(r"[0-5]"))

    sampling_params = SamplingParams(
                    temperature=0.1,
                    max_tokens=1,
                    logits_processors=[logits_processor],
                    )

    outputs = llm.generate([prompt], sampling_params, use_tqdm=True)


    numeric_response = outputs[0].outputs[0].text.strip()

    return numeric_response

selected_row = df_single_statement.iloc[0]
prompt = generate_prompt_fstring(selected_row)

if prompt:
    model_response = get_model_response(prompt)
    print("Risposta numerica del modello:", model_response)
else:
    print("Errore: il prompt non è stato generato correttamente.")


In [ ]:
def get_model_response(prompt):
    logits_processor = build_vllm_logits_processor(tokenizer, RegexParser(r"[0-5]"))

    sampling_params = SamplingParams(
                    temperature=0.1,
                    max_tokens=1,
                    logits_processors=[logits_processor],
                    )

    outputs = llm.generate([prompt], sampling_params, use_tqdm=True)

    responses = []
    for output in outputs:
        responses.append(output.outputs[0].text.strip())

    return responses

selected_row = df_single_statement.iloc[0]
prompt = generate_prompt_fstring(selected_row)

if prompt:
    model_responses = get_model_response(prompt)
    for i, response in enumerate(model_responses):
        print(f"Risposta {i+1}: {response}")
else:
    print("Errore: il prompt non è stato generato correttamente.")
